# 05 &mdash; Graph-level classification, and the expressiveness theorem as an experiment

> **Reading order.** These notebooks are a course, not a reference. Each one
> assumes the previous. `01` builds the tensor vocabulary, `02` builds the layers,
> `03` trains them on Cora, `04` proves they are correct and breaks them
> deliberately, `05` moves to graph-level tasks.
>
> Everything imports the local `gnn_lab` package -- edit the package and re-run
> the cell. The self-contained Kaggle run lives in `kaggle/`.

**Goal.** Move from predicting a label per *node* in one fixed graph to
predicting one label per *whole graph*. That flip forces two new ingredients --
batching and readout -- and it is where the GIN expressiveness result stops
being a theorem and becomes something you can measure.

---

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))     # so `import gnn_lab` works from notebooks/

import torch
import torch.nn as nn
import torch.nn.functional as F

from gnn_lab import (GCNScratch, GraphSAGEScratch, GATScratch, GINScratch,
                     GCNConvScratch, SAGEConvScratch, GATConvScratch, GINConvScratch,
                     load_cora, load_mutag, triangle_square_dataset, karate_club,
                     toy_graph, path_graph, two_communities,
                     train_node, train_node_multi_seed, train_graph_classification,
                     evaluate_node, set_seed, get_device, count_parameters,
                     accuracy, macro_f1, majority_baseline,
                     mean_pairwise_cosine, dirichlet_energy)
from gnn_lab.ops import (scatter_add, scatter_mean, scatter_max, scatter_softmax,
                         gcn_norm, add_self_loops, remove_self_loops, to_undirected,
                         degree, dense_to_edge_index)
from gnn_lab.layers.pooling import collate_graphs, global_pool, graph_batches
from gnn_lab import equivalence, viz

DEVICE = get_device()
RESULTS = os.path.join("..", "results")
os.makedirs(RESULTS, exist_ok=True)

print("torch  :", torch.__version__)
print("device :", DEVICE)
print("PyG    :", "available" if equivalence.PYG_AVAILABLE else f"missing ({equivalence.PYG_IMPORT_ERROR})")
set_seed(0)


## 1. Why this is a different problem

Everything so far used one graph (Cora) and predicted 2708 labels. Now the
dataset *is* a set of graphs, each with a single label:

| | node classification | graph classification |
|---|---|---|
| graphs | 1 | many |
| labels | one per node | one per graph |
| batching | not needed | **required** |
| readout | not needed | **required** |
| split | masks over nodes | folds over graphs |
| metric | accuracy on the test mask | cross-validated accuracy |

Two consequences follow immediately, and both are easy to get wrong.

**Batching.** Graphs have different sizes, so you cannot stack them into a 4-D
tensor the way CNNs stack images. Instead you concatenate everything into one
**block-diagonal graph** plus a `batch` vector recording which graph each node
belongs to.

**Cross-validation.** MUTAG has 188 graphs. A single 80/20 split leaves ~38 test
graphs, so one lucky fold swings accuracy by 5+ points. K-fold uses every graph
for testing exactly once.

In [ ]:
mutag = load_mutag()
print()
sizes = torch.tensor([g["x"].size(0) for g in mutag.graphs], dtype=torch.float)
print(f"nodes per molecule : min {int(sizes.min())}, mean {float(sizes.mean()):.2f}, max {int(sizes.max())}")
print(f"ragged sizes       : {len(set(sizes.tolist()))} distinct values -> cannot be stacked")

### Read the baseline before reading the model

MUTAG is **imbalanced**: 63 negative vs 125 positive. So the majority-class
baseline is **66.5%, not 50%**.

A GIN reporting 65% on MUTAG is worse than predicting a constant. Plenty of
papers have printed numbers like that without noticing.

In [ ]:
y = mutag.labels()
counts = torch.bincount(y, minlength=mutag.num_classes)
print("class counts :", counts.tolist(), "for raw labels", mutag.class_names)
print(f"majority baseline : {mutag.majority_accuracy()*100:.2f}%")
print(f"so anything below {mutag.majority_accuracy()*100:.1f}% is worse than a constant predictor")

## 2. The batching trick, spelled out

Here is the whole idea on three molecules. Concatenate node features,
concatenate edges, and **offset each graph's edge indices** by the running node
count so nothing connects across graphs.

In [ ]:
b = collate_graphs(mutag.graphs[:3])
print(b)
print()
print("batch vector (which molecule each node belongs to):")
print(" ", b.batch.tolist())
print()
print("node counts:", [g["x"].size(0) for g in mutag.graphs[:3]])
print()
print("edges of the batched graph (first 12):")
for s, d in b.edge_index.t().tolist()[:12]:
    print(f"   {s} -> {d}    (molecule {int(b.batch[s])})")
print("   ...")
print()
n0 = mutag.graphs[0]["x"].size(0)
print(f"molecule 1 starts at node index {n0}, so its internal edge (0,1) became "
      f"({n0},{n0+1}).")

### Proof that batching cannot leak information

The molecules are physically disconnected, so a batched forward pass must give
*exactly* the same node embeddings as running each molecule alone. If it does
not, there is a bug in the offset arithmetic -- and this is the test that finds
it.

In [ ]:
mlp = nn.Sequential(nn.Linear(mutag.num_features, 16), nn.ReLU(), nn.Linear(16, 16))
conv = GINConvScratch(mlp).double().eval()

batched = collate_graphs(mutag.graphs[:8])
ok = True
with torch.no_grad():
    joint = conv(batched.x.double(), batched.edge_index, batched.num_nodes)
    ptr = 0
    for k, gr in enumerate(mutag.graphs[:8]):
        n = gr["x"].size(0)
        alone = conv(gr["x"].double(), gr["edge_index"], n)
        same = bool(torch.allclose(joint[ptr:ptr+n], alone, atol=1e-12))
        ok &= same
        print(f"  molecule {k} ({n:2d} nodes): batched == standalone -> {same}")
        ptr += n
print("\nno information leaks between graphs:", ok)

## 3. Readout: collapsing nodes into one vector per graph

Pooling is the *same* scatter primitive as message passing, one level up: "sum
every node whose batch id is k".

The choice is not cosmetic. It changes what the model is able to see:

* **sum** -- degree-sensitive; keeps "how much stuff is in this graph"
* **mean** -- size-invariant; safer when graphs vary a lot in size
* **max** -- keeps only the most strongly activated node per channel, behaving
  like a soft "does this graph contain substructure X?" detector

In [ ]:
x = torch.tensor([[1., 1.],
                  [3., 3.],
                  [5., 5.],
                  [10., 10.]])
batch = torch.tensor([0, 0, 1, 1])
print("x     :", x.tolist())
print("batch :", batch.tolist(), "-> graphs {0,1} and {2,3}")
print()
for m in ("sum", "mean", "max"):
    print(f"  global_pool({m:4}) -> {global_pool(x, batch, m).tolist()}")

## 4. Train GIN on MUTAG

5-fold cross-validation, three readout operators, everything else identical.

In [ ]:
def build_gin(pool):
    return lambda: GINScratch(mutag.num_features, 32, mutag.num_classes,
                              num_layers=3, dropout=0.5, pool=pool).to(DEVICE)

mutag_rows = []
for pool in ("sum", "mean", "max"):
    print(f"=== GIN, {pool} readout ===")
    r = train_graph_classification(build_gin(pool), mutag, model_name=f"GIN-{pool}",
                                   epochs=80, batch_size=32, folds=5, device=DEVICE)
    r["num_parameters"] = count_parameters(build_gin(pool)())
    mutag_rows.append(r)
    print()

In [ ]:
print(f"{'model':<14}{'accuracy':>20}{'macro F1':>20}{'params':>10}")
print("-" * 64)
for r in mutag_rows:
    print(f"{r['model']:<14}{r['acc_mean']*100:>9.2f} +/- {r['acc_std']*100:>5.2f}%"
          f"{r['f1_mean']*100:>11.2f} +/- {r['f1_std']*100:>5.2f}%"
          f"{r['num_parameters']:>10,}")
print("-" * 64)
print(f"{'baseline':<14}{mutag.majority_accuracy()*100:>9.2f}%   (majority class)")
print()
print("MUTAG is small and noisy -- published GIN numbers sit around 75-85% and")
print("the fold-to-fold spread is wide. Treat a 2-point difference here as noise.")

viz.plot_results_bars([{"model": r["model"], "test_acc_mean": r["acc_mean"],
                        "test_acc_std": r["acc_std"]} for r in mutag_rows],
                      title="MUTAG, 5-fold CV (GIN with different readouts)",
                      baseline=mutag.majority_accuracy(),
                      path=f"{RESULTS}/05_mutag.png")

### Does depth help here?

Unlike Cora, graph-level tasks sometimes benefit from more layers, because a
deeper receptive field can capture larger substructures. But over-smoothing still
applies. Measure it rather than assume.

In [ ]:
for layers in (1, 2, 3, 4, 5):
    def build(layers=layers):
        return lambda: GINScratch(mutag.num_features, 32, mutag.num_classes,
                                  num_layers=layers, dropout=0.5, pool="sum").to(DEVICE)
    r = train_graph_classification(build(), mutag, model_name=f"GIN-{layers}L",
                                   epochs=60, batch_size=32, folds=3,
                                   device=DEVICE, verbose=False)
    print(f"  {layers} layers   test {r['acc_mean']*100:5.2f} +/- {r['acc_std']*100:.2f}%")

---
## 5. The expressiveness theorem, as an experiment (and why it needs a caveat)

The GIN paper proves that **mean and max aggregation cannot distinguish certain
non-isomorphic graphs**. They cannot *count*.

Concretely: a node with one neighbour of feature `1` and a node with two
neighbours of feature `0.5` have the same mean and the same max, but different
structure. Sum aggregation tells them apart.

`triangle_vs_square` generates graphs whose label is **"does this graph contain a
3-cycle?"**. Classes are built in matched pairs: same node count, same edge count,
and the positive graph is the negative one with a single edge moved. That matching
matters. An earlier version drew random graphs and rejected the wrong ones, and
**edge density alone predicted the label at 88%** -- so `sum` appeared to win when
it was really just reading density.

**Expected outcome:** every readout near chance. The theorem is about the aggregator
*inside* a layer, this model always sums there, and detecting a triangle is beyond
1-WL message passing anyway. If a clear gap shows up, suspect a shortcut first.

In [ ]:
ts = triangle_square_dataset(num_graphs=400, seed=0)
print()
print("class balance:", torch.bincount(ts.labels()).tolist(),
      f"-> majority baseline {ts.majority_accuracy():.3f}")
sizes = torch.tensor([g["x"].size(0) for g in ts.graphs], dtype=torch.float)
print(f"nodes per graph: mean {float(sizes.mean()):.2f}")

# confirm the labels are what they claim, independently
import itertools
checked = 0
for g in ts.graphs[:25]:
    n = g["x"].size(0)
    adj = [set() for _ in range(n)]
    for a, c in g["edge_index"].t().tolist():
        adj[a].add(c)
    found = any(c in adj[a] for a in range(n) for b in adj[a] for c in adj[b] if c != a)
    assert found == bool(int(g["y"])), "label does not match the actual structure"
    checked += 1
print(f"independently re-verified {checked} labels against the adjacency lists: all correct")

In [ ]:
ts_rows = []
for pool in ("mean", "max", "sum"):
    print(f"=== GIN, {pool} readout, on triangle detection ===")
    def build(pool=pool):
        return lambda: GINScratch(ts.num_features, 32, ts.num_classes,
                                  num_layers=3, dropout=0.5, pool=pool).to(DEVICE)
    r = train_graph_classification(build(), ts, model_name=f"GIN-{pool}",
                                   epochs=60, batch_size=32, folds=3, device=DEVICE)
    ts_rows.append(r)
    print()

In [ ]:
print(f"{'readout':<12}{'accuracy':>20}{'vs chance':>14}")
print("-" * 46)
for r in ts_rows:
    print(f"{r['model']:<12}{r['acc_mean']*100:>9.2f} +/- {r['acc_std']*100:>5.2f}%"
          f"{(r['acc_mean']-0.5)*100:>+13.2f}")
print("-" * 46)
print(f"{'chance':<12}{50.0:>9.2f}%")

gap = next(r["acc_mean"] for r in ts_rows if r["model"] == "GIN-sum") -       next(r["acc_mean"] for r in ts_rows if r["model"] == "GIN-mean")
print(f"\nsum - mean readout gap: {gap*100:+.2f} points")
print("Expected clearly positive. This is the Weisfeiler-Lehman expressiveness")
print("bound showing up as a number on a real task, not as a theorem in a paper.")

viz.plot_results_bars([{"model": r["model"], "test_acc_mean": r["acc_mean"],
                        "test_acc_std": r["acc_std"]} for r in ts_rows],
                      title="Triangle detection: readout pooling decides the outcome",
                      baseline=0.5, path=f"{RESULTS}/05_triangle.png")

### Why this is a sharper test than MUTAG

MUTAG differences of 2 points are within noise. Here the *mechanism* is what is
under test: if `sum` beats `mean` by a wide margin on a task constructed so that
counting is the only signal, the implementation is expressing what the theory
says it should. If the two readouts tie, either the task is too easy, or the
pooling is not wired to the batch vector correctly -- and the leak test in
section 2 is what tells you which.

## 6. Optional: a larger real dataset

PROTEINS is 1113 protein structures (enzyme vs non-enzyme), ~39 nodes each. Same
pipeline, no code changes -- only the dataset name. It trains in well under a
minute on a T4.

Uncomment to run.

In [ ]:
# proteins = load_proteins()
# print()
# def build_p(pool="sum"):
#     return lambda: GINScratch(proteins.num_features, 64, proteins.num_classes,
#                               num_layers=4, dropout=0.5, pool=pool).to(DEVICE)
# r = train_graph_classification(build_p(), proteins, model_name="GIN-PROTEINS",
#                                epochs=60, batch_size=64, folds=5, device=DEVICE)
# print(f"\nPROTEINS: {r['acc_mean']*100:.2f} +/- {r['acc_std']*100:.2f}% "
#       f"(baseline {r['majority_baseline']*100:.2f}%)")

## 7. Summary

* **Graph-level tasks need two things node tasks do not**: a batching scheme for
  ragged graphs (block-diagonal + `batch` vector) and a readout that collapses
  nodes into one vector per graph.
* **Pooling is the same scatter primitive as message passing**, one level up.
  There is no new machinery.
* **Batching is provably safe**: the leak test shows batched and standalone
  forward passes agree to `1e-12`.
* **Always print the baseline.** MUTAG's is 66.5%, not 50%.
* **A result that matches your story is the one to check hardest.** The first
  triangle experiment showed `sum` beating `mean`; it was an edge-density leak.
  With classes matched on size and density, every readout is near chance.

---

## Where to go from here

| direction | why |
|---|---|
| **Neighbour sampling** | `gnn_lab` is full-batch. Sampling is what makes large graphs trainable, and SAGE was designed for it. |
| **Heterophilous graphs** | Cora and MUTAG are homophilous. On graphs where neighbours *differ*, GCN fails and the fix (separating self and neighbour transforms, or attention) becomes essential rather than optional. |
| **Graph transformers** | GPS / GraphGPS remove the locality constraint entirely. Worth understanding once the receptive-field argument here is intuitive. |
| **Edge features** | `gnn_lab` ignores `edge_attr`. MUTAG ships bond types; using them is a natural extension. |
| **Link prediction** | The third canonical task. Needs a decoder on top of node embeddings and a negative-sampling scheme. |